<a href="https://colab.research.google.com/github/KhamidjonovRakhmatullo/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
print("Menu Detector")

Menu Detector


In [7]:
from google.colab import drive
import torchvision.transforms as transforms

# PIL = python image library
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset
import os
import numpy as np

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# Dataset main folder path in Google Drive
# This folder contains our food image folders:
# cheesecake, chocolate_cake, hot_dog, kebab, pilaf, pizza

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print("\nDataset path:", DATASET_PATH)


# Label grouping / class consolidation
# This dictionary maps original folder names to final class names.
#
# Format:
# "original_folder_name": "final_class_name"
#
# Here, cheesecake and chocolate_cake will be grouped as one class: dessert

CUSTOM_CLASS_MAPPING = {
    'cheesecake': 'dessert',
    'chocolate_cake': 'dessert',
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'kebab': 'kebab',
    'pilaf': 'pilaf',
    'pizza': 'pizza'
}


# Final class names that our model will learn
# cheesecake and chocolate_cake are not separate classes now.
# They are combined into one class: dessert

CLASSES = [
    'dessert',
    'hamburger',
    'hot_dog',
    'kebab',
    'pilaf',
    'pizza'
]
print("\nClasses:", CLASSES)


# Count how many final classes we have
NUM_CLASSES = len(CLASSES)
print("\nClass length:", NUM_CLASSES)


# Convert class names into numbers
# AI models work with numbers, not text labels.
#
# Result will be:
# {
#   'dessert': 0,
#   'hamburger': 1,
#   'hot_dog': 2,
#   'kebab': 3,
#   'pilaf': 4,
#   'pizza': 5
# }

CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
print("\nCLASS INDEX:", CLASS_TO_IDX)

# ------------
# Create image transformation pipeline
# This pipeline prepares each image before giving it to the model

transform = transforms.Compose([

    # Resize every image to 224x224 pixels
    # Models need images with the same size
    transforms.Resize((224, 224)),

    # Convert image to PyTorch tensor
    # Also changes pixel values from 0-255 to 0.0-1.0
    transforms.ToTensor(),

    # Normalize image pixel values for each RGB channel
    # Formula: new_value = (old_value - mean) / std
    #
    # mean and std values are commonly used for pretrained ImageNet models
    #
    # Red channel:   (value - 0.485) / 0.229
    # Green channel: (value - 0.456) / 0.224
    # Blue channel:  (value - 0.406) / 0.225
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


Dataset path: /content/drive/MyDrive/food101_dataset

Classes: ['dessert', 'hamburger', 'hot_dog', 'kebab', 'pilaf', 'pizza']

Class length: 6

CLASS INDEX: {'dessert': 0, 'hamburger': 1, 'hot_dog': 2, 'kebab': 3, 'pilaf': 4, 'pizza': 5}


CUSTOM_CLASS_MAPPING -> rename/group folder labels

CLASSES -> final categories for the model

CLASS_TO_IDX -> convert class names to numbers for model training

In [21]:
#------------------------------
# Custom Dataset Class
#------------------------------

# FoodDataset class inherits from PyTorch Dataset
# This class helps us load images and labels one by one for model training

class FoodDataset(Dataset):

    # __init__ runs when we create dataset object
    # images -> list of image file paths
    # labels -> list of image labels
    # transform -> image preprocessing pipeline, for example Resize, ToTensor, Normalize
    def __init__(self, images, labels, transform=None):
        self.images = images          # Store image paths
        self.labels = labels          # Store labels
        self.transform = transform    # Store transform pipeline
        # Store will not write index: "0": "image_path_1" | it writes only "image_path_1"
        # Python list automatically has index.
        # Store shape:
        # images = [image_path_1, image_path_2, image_path_3]
        # labels = [label_1, label_2, label_3]


    # __len__ returns total number of images in dataset
    # PyTorch uses this to know dataset size
    def __len__(self):
        print('images_length:', len(self.images))
        return len(self.images)


    # __getitem__ gets one image and its label by index
    # PyTorch DataLoader calls this method automatically
    def __getitem__(self, idx):

        # Get image path by index
        img_path = self.images[idx]
        print("image_path:", img_path)


        # Get label by same index
        # Example:
        # image path -> images[0]
        # label      -> labels[0]
        label = self.labels[idx]
        print('label:', label)


        # Try to open image file
        # convert('RGB') makes sure image has 3 color channels: Red, Green, Blue
        try:
            image = Image.open(img_path).convert('RGB')


        # If image is broken or cannot be opened, skip it
        # Then load next image instead
        except (UnidentifiedImageError, OSError):
            print(f'Skipping broken image: {img_path}')
            return self.__getitem__((idx + 1) % len(self.images))


        # If transform exists, apply it to the image
        # Example:
        # Resize -> ToTensor -> Normalize
        if self.transform:
            image = self.transform(image)


        # Return final image and label
        # This is what model receives during training
        return image, label

In [23]:
# ============================
# Gather and Split Data
# ============================

# This list will store all image paths with their labels
# Each item will look like this:
# ("image_path", label_number)
all_images = []


# Loop through custom class mapping
# original_class -> folder name in Google Drive         | cheesecake
# mapped_class   -> final class name used by the model. | dessert
#   cheesecake      dessert        {'cheesecake': 'dessert'.....}
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():

    # Create full path for each class folder
    # Example:
    # /content/drive/MyDrive/food101_dataset/cheesecake
    class_path = os.path.join(DATASET_PATH, original_class)
    print("class_path:", class_path)

    # Check if this class folder exists
    # If folder does not exist, skip it
    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue


    # Loop through all image files inside the class folder
    for img in os.listdir(class_path):

        # Only use image files with these extensions
        if img.endswith(('.jpg', '.jpeg', '.png')):

            # Create full image path
            # Example:
            # /content/drive/MyDrive/food101_dataset/pizza/image1.jpg
            full_path = os.path.join(class_path, img)

            # Get label number from mapped class name
            # Example:
            # mapped_class = "pizza"
            # CLASS_TO_IDX["pizza"]  | label = 4
            label = CLASS_TO_IDX[mapped_class]

            # Add image path and label together
            # This keeps image and label matched by same index
            all_images.append((full_path, label))


# Shuffle all images randomly
# This helps mix classes before splitting train/validation data
np.random.shuffle(all_images)

# Split 80% for training and 20% for validation
split = int(0.8 * len(all_images))

# First 80% data for training
train_data = all_images[:split]

# Last 20% data for validation
val_data = all_images[split:]


# Separate image paths and labels for training data
# train_images -> image paths
# train_labels -> label numbers
train_images, train_labels = zip(*train_data)

# Separate image paths and labels for validation data
# val_images -> image paths
# val_labels -> label numbers
val_images, val_labels = zip(*val_data)


# Print all image-label pairs to check
print("first 3 images:", all_images[:3])


# Create dataset object with training (image paths |and| labels)
# FoodDataset stores:
# self.images = train_images
# self.labels = train_labels
dataset = FoodDataset(train_images, train_labels)

# Print total number of training images
# This calls __len__()
print(len(dataset))

# Get one image-label pair by index
# 291 is item index, not label number
# This calls __getitem__(291)
#
# Example:
# dataset[291] -> self.images[291] + self.labels[291]
img, lbl = dataset[291]

class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
class_path: /content/drive/MyDrive/food101_dataset/pizza
first 3 images: [('/content/drive/MyDrive/food101_dataset/chocolate_cake/2425586.jpg', 0), ('/content/drive/MyDrive/food101_dataset/pizza/1069629.jpg', 5), ('/content/drive/MyDrive/food101_dataset/hot_dog/91857.jpg', 2)]
images_length: 4160
4160
image_path: /content/drive/MyDrive/food101_dataset/chocolate_cake/2531666.jpg
label: 0
